# Preprocessing and DistilBERT Model

This notebook demonstrates how raw AG News text becomes model input. The full reusable pipeline is in `src/data_preprocessing.py` and `src/model.py`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import LABEL_NAMES, MAX_LENGTH, MODEL_NAME
from src.data_preprocessing import (
    create_dataloader, get_tokenizer, prepare_dataframes, tokenize_dataframes
)
from src.model import DistilBertNewsClassifier

## 1. Clean and split the data

The cleaning is intentionally light: repair HTML artifacts, normalize
whitespace, and preserve normal sentence structure. We do not remove stop words,
stem words, or lemmatize because DistilBERT uses word order and context.

After removing 137 exact training duplicates, a stratified 90/10 split creates
107,876 training rows and 11,987 validation rows. The 7,600-row official test
split remains untouched.

In [2]:
frames = prepare_dataframes()
split_sizes = {name: len(frame) for name, frame in frames.items()}
display(pd.Series(split_sizes, name='Rows'))
display(frames['train'].head())

train         107876
validation     11987
test            7600
Name: Rows, dtype: int64

,text,label
0,"Jeter, Yankees Look Dashing Once Again Here it...",1
1,Busch sweeps Loudon: No. 19 takes lead in Next...,1
2,GB badminton duo make final Nathan Robertson a...,1
3,Arms embargo imposed on Ivory Coast Villagers ...,0
4,Russia Urged to Combat Hazing in Army (AP) AP ...,0


## 2. Inspect one tokenized example

DistilBERT uses subword tokenization. `input_ids` identify tokens and `attention_mask` distinguishes real tokens from padding.

In [3]:
tokenizer = get_tokenizer()
sample_text = frames['train'].iloc[0]['text']
sample_label = int(frames['train'].iloc[0]['label'])
encoded = tokenizer(
    sample_text, truncation=True, max_length=MAX_LENGTH, padding='max_length'
)

print('Original text:', sample_text)
print('Correct label:', LABEL_NAMES[sample_label])
print('First 25 tokens:', tokenizer.convert_ids_to_tokens(encoded['input_ids'][:25]))
print('First 25 input IDs:', encoded['input_ids'][:25])
print('First 25 attention values:', encoded['attention_mask'][:25])
print('Sequence length:', len(encoded['input_ids']))

Original text: Jeter, Yankees Look Dashing Once Again Here it comes. You can feel the rumbling of anticipation now. Just two games into the American League playoffs, we already have a colossal collision coming into focus.
Correct label: Sports
First 25 tokens: ['[CLS]', 'jet', '##er', ',', 'yankees', 'look', 'dash', '##ing', 'once', 'again', 'here', 'it', 'comes', '.', 'you', 'can', 'feel', 'the', 'rumbling', 'of', 'anticipation', 'now', '.', 'just', 'two']
First 25 input IDs: [101, 6892, 2121, 1010, 11081, 2298, 11454, 2075, 2320, 2153, 2182, 2009, 3310, 1012, 2017, 2064, 2514, 1996, 26670, 1997, 11162, 2085, 1012, 2074, 2048]
First 25 attention values: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Sequence length: 128


## 3. Create a PyTorch batch

For a quick demonstration, only 32 rows from each split are tokenized below. Change `PREPARE_FULL_DATASET` to `True` to process every row in this notebook. The training script always uses the complete dataset.

In [4]:
PREPARE_FULL_DATASET = False
frames_to_tokenize = frames if PREPARE_FULL_DATASET else {
    name: frame.head(32).copy() for name, frame in frames.items()
}
tokenized = tokenize_dataframes(frames_to_tokenize, tokenizer, max_length=MAX_LENGTH)
demo_loader = create_dataloader(tokenized['train'], tokenizer, batch_size=8, shuffle=False)
batch = next(iter(demo_loader))
for name, tensor in batch.items():
    print(f'{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}')

Tokenizing train:   0%|          | 0/32 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/32 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/32 [00:00<?, ? examples/s]

input_ids: shape=(8, 62), dtype=torch.int64
attention_mask: shape=(8, 62), dtype=torch.int64
labels: shape=(8,), dtype=torch.int64


## 4. Pass the batch through the model

DistilBERT creates a contextual vector for each token. We use the first-token vector, apply dropout, and send it through a linear layer with four outputs. The four raw outputs are logits; softmax converts them to probabilities.

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DistilBertNewsClassifier(MODEL_NAME).to(device)
model.eval()

with torch.no_grad():
    logits = model(
        input_ids=batch['input_ids'].to(device),
        attention_mask=batch['attention_mask'].to(device),
    )
    probabilities = torch.softmax(logits, dim=1)

print('Device:', device)
print('Logit shape:', tuple(logits.shape))
print('First example probabilities:')
display(pd.Series(probabilities[0].cpu().numpy(), index=LABEL_NAMES, name='Probability'))

Device: cuda
Logit shape: (8, 4)
First example probabilities:


World       0.231295
Sports      0.180167
Business    0.264546
Sci/Tech    0.323992
Name: Probability, dtype: float32

## 5. How training changes the model

The demonstration above initializes pretrained DistilBERT and a new four-output
linear layer. Before fine-tuning, the class probabilities are not meaningful.
During training:

1. Cross-entropy loss compares the logits with the correct class.
2. Backpropagation calculates gradients.
3. AdamW updates the DistilBERT and classification-layer weights.
4. The attention mask prevents padding tokens from affecting the representation.
5. The checkpoint with the lowest validation loss is saved.

This notebook uses a small sample for explanation. `src/train.py` always trains
on the complete training split.